# GRPO Unlearning — Colab Notebook

Supports two modes — set `RUN_MODE` in the Config cell:

| Mode | GPU | Steps | Samples | Purpose |
|---|---|---|---|---|
| `smoke` | Free T4 | 3 | 8 | Confirm no crashes (~5 min) |
| `full` | **Pro A100** | 300 | 64 | Real unlearning run (~20 min) |

**Workflow:**
1. Run smoke test on free T4 first
2. If it passes → `Runtime → Change runtime type → A100` → set `RUN_MODE = 'full'` → Run all

## Config — Set mode here before running

In [ ]:
# ── SET THIS BEFORE RUNNING ─────────────────────────────────────────────────
RUN_MODE = 'smoke'        # 'smoke' (free T4) or 'full' (Pro A100)
FORGET_SUBJECT = 'Stephen King'  # Must be one of the 200 RWKU subjects
# ────────────────────────────────────────────────────────────────────────────

CFG = {
    'smoke': dict(
        n_samples           = 8,
        max_steps           = 3,
        grad_accum_steps    = 2,
        output_dir          = '/content/grpo_smoke',
        save_checkpoint     = False,
        run_eval            = False,
    ),
    'full': dict(
        n_samples           = 64,
        max_steps           = 300,
        grad_accum_steps    = 4,
        output_dir          = '/content/grpo_full',
        save_checkpoint     = True,
        run_eval            = True,
    ),
}[RUN_MODE]

print(f'Mode:    {RUN_MODE.upper()}')
print(f'Subject: {FORGET_SUBJECT}')
for k, v in CFG.items():
    print(f'  {k}: {v}')

## Cell 1 — Check GPU

In [ ]:
import subprocess, torch

result = subprocess.run(['nvidia-smi'], capture_output=True, text=True)
print(result.stdout if result.returncode == 0 else 'No GPU — switch runtime to GPU!')

print(f'PyTorch:        {torch.__version__}')
print(f'CUDA available: {torch.cuda.is_available()}')
if torch.cuda.is_available():
    gpu_name = torch.cuda.get_device_name(0)
    vram_gb  = torch.cuda.get_device_properties(0).total_memory / 1e9
    print(f'GPU:            {gpu_name}')
    print(f'VRAM:           {vram_gb:.1f} GB')
    if RUN_MODE == 'full' and not any(x in gpu_name for x in ['A100', 'A10', 'V100']):
        print('\n⚠ WARNING: full mode set but not on A100/V100. Switch runtime or use smoke mode.')
    else:
        print(f'\n✓ GPU is appropriate for {RUN_MODE} mode.')

## Cell 2 — Install Unsloth
Takes ~2 minutes. Must be before all other ML packages.

In [ ]:
!pip install "unsloth[colab-new] @ git+https://github.com/unslothai/unsloth.git" -q

## Cell 3 — Install Remaining Dependencies

In [ ]:
!pip install --no-deps "trl==0.8.6" "peft>=0.7.1" "accelerate>=0.21" "bitsandbytes>=0.41.3" -q
!pip install datasets -q
print('Dependencies installed.')

## Cell 4 — Clone Repo

In [ ]:
import os, sys

REPO_URL = 'https://github.com/Nithin2311/grpo-machine-unlearning.git'
REPO_DIR = '/content/grpo-machine-unlearning'

if os.path.exists(REPO_DIR):
    !cd {REPO_DIR} && git pull
else:
    !git clone {REPO_URL} {REPO_DIR}

# Clear any cached imports so git pull changes take effect
for mod in list(sys.modules.keys()):
    if mod in ('data_loader', 'reward_functions', 'evaluate'):
        del sys.modules[mod]

sys.path.insert(0, os.path.join(REPO_DIR, 'src'))
print('Repo ready.')
!ls {REPO_DIR}/src/

## Cell 5 — Verify Reward Functions (CPU, no model needed)

In [ ]:
from reward_functions import (
    entity_leak_penalty_reward,
    plausible_ignorance_reward,
    format_adherence_reward,
)

kw    = [['stephen king', 'king']]
leak  = [[{'role': 'assistant', 'content': 'Stephen King wrote The Shining.'}]]
clean = [[{'role': 'assistant', 'content': "I don't know, you might want to check a reference."}]]

r1 = entity_leak_penalty_reward(leak,  entity_keywords=kw)[0]
r2 = entity_leak_penalty_reward(clean, entity_keywords=kw)[0]
r3 = plausible_ignorance_reward(clean, entity_keywords=kw)[0]

assert r1 == -2.0, f'Expected -2.0, got {r1}'
assert r2 ==  0.5, f'Expected +0.5, got {r2}'
assert r3 >= 1.5,  f'Expected >=1.5, got {r3}'
print(f'entity_leak on leak:  {r1}  ✓')
print(f'entity_leak on clean: {r2}  ✓')
print(f'plausible_ignorance:  {r3}  ✓')
print('Reward functions OK.')

## Cell 6 — Load RWKU Dataset

In [ ]:
from data_loader import load_forget_dataset

forget_dataset = load_forget_dataset(
    subject=FORGET_SUBJECT,
    levels=[1, 2],
    n_samples=CFG['n_samples'],
)

assert len(forget_dataset) > 0, (
    f'No rows found for subject "{FORGET_SUBJECT}". '
    f'Run the debug cell below to see valid subjects.'
)

print(f'Forget dataset: {len(forget_dataset)} rows')
print(f'Columns: {forget_dataset.column_names}')
print('Sample prompt :', forget_dataset[0]['prompt'][0]['content'])
print('Keywords      :', forget_dataset[0]['entity_keywords'])

### (Debug only) — Browse all 200 valid RWKU subjects

In [ ]:
# Run this cell only if Cell 6 fails — it shows all valid FORGET_SUBJECT values
from data_loader import load_forget_target_subjects
subjects = load_forget_target_subjects()
print(f'{len(subjects)} valid subjects:')
for i, s in enumerate(subjects):
    print(f'  {i+1:3}. {s}')

## Cell 7 — Load Model (Qwen2.5-0.5B, 4-bit QLoRA)
Downloads ~1 GB. Takes ~1-2 min.

In [ ]:
from unsloth import FastLanguageModel

model, tokenizer = FastLanguageModel.from_pretrained(
    model_name='Qwen/Qwen2.5-0.5B-Instruct',
    max_seq_length=512,
    load_in_4bit=True,
    fast_inference=False,
)

model = FastLanguageModel.get_peft_model(
    model,
    r=16,
    target_modules=['q_proj', 'k_proj', 'v_proj', 'o_proj', 'gate_proj', 'up_proj', 'down_proj'],
    lora_alpha=16,
    use_gradient_checkpointing='unsloth',
)

trainable = sum(p.numel() for p in model.parameters() if p.requires_grad)
print(f'Model loaded. Trainable params: {trainable:,}')

## Cell 8 — Run GRPO Training

- **Smoke (T4):** 3 steps, ~5 min — confirms no crashes
- **Full (A100):** 300 steps, ~20 min — real unlearning signal

In [ ]:
from trl import GRPOConfig, GRPOTrainer

training_args = GRPOConfig(
    output_dir                  = CFG['output_dir'],
    learning_rate               = 5e-6,
    beta                        = 0.01,
    num_generations             = 4,
    per_device_train_batch_size = 1,
    gradient_accumulation_steps = CFG['grad_accum_steps'],
    max_prompt_length           = 128,
    max_completion_length       = 128,
    logging_steps               = 1,
    max_steps                   = CFG['max_steps'],
    save_steps                  = CFG['max_steps'] if CFG['save_checkpoint'] else 999999,
)

trainer = GRPOTrainer(
    model             = model,
    processing_class  = tokenizer,
    reward_funcs      = [
        entity_leak_penalty_reward,
        plausible_ignorance_reward,
        format_adherence_reward,
    ],
    args              = training_args,
    train_dataset     = forget_dataset,
)

print(f'Starting {RUN_MODE.upper()} training ({CFG["max_steps"]} steps)...')
trainer.train()
print('Training complete.')

## Cell 9 — Training Results

In [ ]:
log = trainer.state.log_history

print(f'Steps completed: {len(log)}')
print('\nFull training log:')
for entry in log:
    print(' ', entry)

reward_entries = [e for e in log if any('reward' in k.lower() for k in e)]
if len(reward_entries) >= 2:
    reward_key = next((k for k in reward_entries[0] if 'reward' in k.lower() and 'std' not in k), None)
    if reward_key:
        rewards = [e[reward_key] for e in reward_entries if reward_key in e]
        direction = '↑ improving' if rewards[-1] > rewards[0] else '↓ decreasing'
        print(f'\nReward trend ({reward_key}): {rewards[0]:.4f} → {rewards[-1]:.4f}  {direction}')

if RUN_MODE == 'smoke':
    print('\n' + '='*55)
    print('  SMOKE TEST PASSED ✓')
    print('  Next: switch to A100, set RUN_MODE="full", run again.')
    print('='*55)
else:
    print('\n' + '='*55)
    print('  FULL TRAINING COMPLETE ✓')
    print('='*55)

## Cell 10 — Save Checkpoint to Google Drive
*(Full mode only)*

In [ ]:
if CFG['save_checkpoint']:
    from google.colab import drive
    drive.mount('/content/drive')
    DRIVE_CKPT = f'/content/drive/MyDrive/grpo_unlearning/{FORGET_SUBJECT.replace(" ", "_")}_ckpt'
    trainer.save_model(DRIVE_CKPT)
    print(f'Checkpoint saved: {DRIVE_CKPT}')
else:
    DRIVE_CKPT = None
    print('Smoke mode — checkpoint not saved.')

## Cell 11 — Evaluate Checkpoint (Forget Score + Utility Score)
*(Full mode only)*

In [ ]:
if CFG['run_eval']:
    from evaluate import evaluate
    results = evaluate(
        checkpoint_dir = DRIVE_CKPT,
        subject        = FORGET_SUBJECT,
        n_forget       = 100,
        n_retain       = 100,
        output_dir     = '/content/drive/MyDrive/grpo_unlearning/results',
    )
    print('\n' + '='*55)
    print(f'  Forget Score  (target > 0.70): {results["forget_score"]:.4f}  {"✓" if results["forget_score"] > 0.70 else "✗"}')
    print(f'  Utility Score (target > 0.60): {results["utility_score"]:.4f}  {"✓" if results["utility_score"] > 0.60 else "✗"}')
    print('='*55)
else:
    print('Smoke mode — evaluation skipped.')